In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/DL_Final_Project"

import os
import sys

sys.path.extend([os.getcwd(), os.path.join(os.getcwd(), "notebook")])

print(os.listdir("/content/drive/MyDrive/DL_Final_Project"))

['weather_data', 'electricity_data', 'supervised.ipynb', 'Fine_Tuning.ipynb', 'Lin_Prob.ipynb', 'main.ipynb']


In [ ]:
import pandas as pd

WEATHER_DIR = f"{BASE_DIR}/weather_data"

files = [f for f in os.listdir(WEATHER_DIR)]
print(files)

path = os.path.join(WEATHER_DIR, files[0])
df = pd.read_csv(path, encoding="latin1")

print(df.shape)
display(df.head())
print(df.dtypes)

['mpi_roof_2025.csv']
(52560, 22)


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),...,wv (m/s),max. wv (m/s),wd (deg),rain (mm),raining (s),SWDR (W/m²),PAR (µmol/m²/s),max. PAR (µmol/m²/s),Tlog (degC),CO2 (ppm)
0,01.01.2025 00:10:00,997.30,-1.56,271.80,-2.97,90.0,5.45,4.90,0.54,3.06,...,1.61,3.87,148.9,0.0,0.0,0.0,0.0,0.0,6.24,456.9
1,01.01.2025 00:20:00,997.05,-0.90,272.48,-2.66,87.8,5.72,5.02,0.70,3.14,...,1.71,3.05,145.6,0.0,0.0,0.0,0.0,0.0,6.32,451.7
2,01.01.2025 00:30:00,996.96,-0.51,272.88,-2.44,86.7,5.89,5.10,0.78,3.19,...,1.61,3.31,146.9,0.0,0.0,0.0,0.0,0.0,6.46,451.0
3,01.01.2025 00:40:00,996.77,-0.27,273.14,-2.26,86.3,5.99,5.17,0.82,3.23,...,1.86,2.93,137.0,0.0,0.0,0.0,0.0,0.0,6.63,450.4
4,01.01.2025 00:50:00,996.62,-0.09,273.33,-2.14,86.0,6.07,5.22,0.85,3.26,...,1.25,2.23,131.3,0.0,0.0,0.0,0.0,0.0,6.82,450.4


Date Time                object
p (mbar)                float64
T (degC)                float64
Tpot (K)                float64
Tdew (degC)             float64
rh (%)                  float64
VPmax (mbar)            float64
VPact (mbar)            float64
VPdef (mbar)            float64
sh (g/kg)               float64
H2OC (mmol/mol)         float64
rho (g/m**3)            float64
wv (m/s)                float64
max. wv (m/s)           float64
wd (deg)                float64
rain (mm)               float64
raining (s)             float64
SWDR (W/m²)             float64
PAR (µmol/m²/s)         float64
max. PAR (µmol/m²/s)    float64
Tlog (degC)             float64
CO2 (ppm)               float64
dtype: object


In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.isna().sum().sort_values(ascending=False).head(20)

(52560, 22)
['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m²)', 'PAR (µmol/m²/s)', 'max. PAR (µmol/m²/s)', 'Tlog (degC)', 'CO2 (ppm)']
Date Time                object
p (mbar)                float64
T (degC)                float64
Tpot (K)                float64
Tdew (degC)             float64
rh (%)                  float64
VPmax (mbar)            float64
VPact (mbar)            float64
VPdef (mbar)            float64
sh (g/kg)               float64
H2OC (mmol/mol)         float64
rho (g/m**3)            float64
wv (m/s)                float64
max. wv (m/s)           float64
wd (deg)                float64
rain (mm)               float64
raining (s)             float64
SWDR (W/m²)             float64
PAR (µmol/m²/s)         float64
max. PAR (µmol/m²/s)    float64
Tlog (degC)           

,0
Date Time,0
p (mbar),0
T (degC),0
Tpot (K),0
Tdew (degC),0
rh (%),0
VPmax (mbar),0
VPact (mbar),0
VPdef (mbar),0
sh (g/kg),0


In [ ]:
import os
import sys

sys.path.extend([os.getcwd(), os.path.join(os.getcwd(), "notebook")])

from helpers.data import load_weather, load_electricity, split_and_scale

Bad timestamps: 0
0   2025-01-01 00:10:00
1   2025-01-01 00:20:00
2   2025-01-01 00:30:00
3   2025-01-01 00:40:00
4   2025-01-01 00:50:00
Name: Date Time, dtype: datetime64[ns]
Date Time
0 days 00:10:00    52559
Name: count, dtype: int64


In [ ]:
# 1. Define the split ratios
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.8) # 70% to 80% is the 10% validation slice

# 2. Split the dataframe
# We keep the raw data for now; we'll handle normalization next
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"Total records: {n}")
print(f"Train set: {len(train_df)} rows")
print(f"Val set:   {len(val_df)} rows")
print(f"Test set:  {len(test_df)} rows")

# 3. Separate features from the timestamp
# The 'Date Time' column isn't a feature the model processes directly
# We'll need it for plotting later, but for the Transformer, we focus on the values.
cols_to_drop = [time_col]
features = [c for c in df.columns if c not in cols_to_drop]

print(f"\nNumber of features to be used: {len(features)}")

Total records: 52560
Train set: 36792 rows
Val set:   5256 rows
Test set:  10512 rows

Number of features to be used: 21


In [ ]:
from helpers.datasets import WeatherDataset
from helpers.models import PatchEmbedding, RevIN, PatchTST_SelfSupervised, PatchTST_Supervised
import torch
from torch.utils.data import DataLoader

# --- Implementation Example ---
# Assuming 'train_df', 'val_df', and 'test_df' are ready from your preprocessing block
# and 'features' contains the list of column names you are using.

seq_len = 512  # Set by the paper for representation learning experiments
pred_len = 96  # Start with T=96 as per Table 4

# Initialize datasets
train_dataset = WeatherDataset(train_df[features], seq_len, pred_len)
val_dataset = WeatherDataset(val_df[features], seq_len, pred_len)
test_dataset = WeatherDataset(test_df[features], seq_len, pred_len)

# Initialize dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 282


In [ ]:
from helpers.training import pretrain_model
from helpers.training import EarlyStopping
from helpers.training import pretrain_model_with_es
from helpers.training import linear_probing_with_es

import torch
import pandas as pd

pred_lengths = [96, 192, 336, 720]
seq_len = 512
batch_size = 128
results_summary = {"Horizon": [], "Val MSE": [], "Val MAE": []}

print("=== PHASE 1: Self-Supervised Pre-Training (Weather) ===")
train_dataset_ssl = WeatherDataset(train_df[features], seq_len, pred_len=96)
val_dataset_ssl = WeatherDataset(val_df[features], seq_len, pred_len=96)
train_loader_ssl = DataLoader(train_dataset_ssl, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader_ssl = DataLoader(val_dataset_ssl, batch_size=batch_size, shuffle=False)

model_ssl = PatchTST_SelfSupervised(num_features=len(features), seq_len=seq_len, patch_len=12, stride=12)
# Using pretrain_model_with_es to handle the 100 epoch limit efficiently
trained_ssl_model = pretrain_model_with_es(model_ssl, train_loader_ssl, val_loader_ssl, epochs=100, lr=1e-4, patience=10)

print("\n=== PHASE 2: Linear Probing (Weather) ===")
for pred_len in pred_lengths:
    print(f"\n--- Horizon: T={pred_len} ---")
    train_dataset = WeatherDataset(train_df[features], seq_len, pred_len)
    val_dataset = WeatherDataset(val_df[features], seq_len, pred_len)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model_sup = PatchTST_Supervised(num_features=len(features), seq_len=seq_len, pred_len=pred_len, patch_len=12, stride=12)
    trained_sup_model = linear_probing_with_es(model_sup, trained_ssl_model, train_loader, val_loader, epochs=20, lr=1e-4, patience=5)

    trained_sup_model.eval()
    mse_crit, mae_crit = nn.MSELoss(), nn.L1Loss()
    v_mse, v_mae = 0.0, 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            preds = trained_sup_model(bx)
            v_mse += mse_crit(preds, by).item()
            v_mae += mae_crit(preds, by).item()

    results_summary["Horizon"].append(pred_len)
    results_summary["Val MSE"].append(v_mse / len(val_loader))
    results_summary["Val MAE"].append(v_mae / len(val_loader))

print("\nFINAL RESULTS (WEATHER)")
print(pd.DataFrame(results_summary).set_index("Horizon").round(3))

=== PHASE 1: Self-Supervised Pre-Training (Weather) ===


Epoch 1/100 [Pre-train]: 100%|██████████| 282/282 [00:58<00:00,  4.79it/s]


Epoch 1 | Train Loss: 0.7937 | Val Loss: 0.4596


Epoch 2/100 [Pre-train]: 100%|██████████| 282/282 [00:58<00:00,  4.83it/s]


Epoch 2 | Train Loss: 0.4016 | Val Loss: 0.3264


...

Epoch 18 | Train MSE: 20162173381606.1211 | Val MSE: 6389162917888.0000 | Val MAE: 631669.3242
EarlyStopping counter: 3 out of 5


Epoch 19/20 [Linear Probing]: 100%|██████████| 277/277 [00:20<00:00, 13.53it/s]


Epoch 19 | Train MSE: 15362177284783.5957 | Val MSE: 4193777457152.0000 | Val MAE: 535571.0430
EarlyStopping counter: 4 out of 5


Epoch 20/20 [Linear Probing]: 100%|██████████| 277/277 [00:20<00:00, 13.55it/s]


Epoch 20 | Train MSE: 18312251350773.8320 | Val MSE: 3161693235200.0000 | Val MAE: 443612.6704

FINAL RESULTS (WEATHER)
              Val MSE     Val MAE
Horizon                          
96       3.118930e+12  453282.287
192      2.146466e+12  369374.516
336      9.753807e+12  674987.046
720      3.161693e+12  443612.670


In [ ]:
print("=== PREPROCESSING ELECTRICITY DATA ===")
ELECTRICITY_DIR = f"{BASE_DIR}/electricity_data"
file_path = os.path.join(ELECTRICITY_DIR, "LD2011_2014.txt")
df_elec_raw = pd.read_csv(file_path, sep=';', decimal=',', low_memory=False)
time_col = df_elec_raw.columns[0]
df_elec = df_elec_raw.copy()
df_elec[time_col] = pd.to_datetime(df_elec[time_col].astype(str).str.strip(), errors="coerce")
df_elec = df_elec.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

elec_train_end = int(len(df_elec) * 0.7)
elec_val_end = int(len(df_elec) * 0.8)
train_df_elec = df_elec.iloc[:elec_train_end]
val_df_elec = df_elec.iloc[elec_train_end:elec_val_end]
elec_features = [c for c in df_elec.columns if c != time_col]

print("\n=== PHASE 1: Self-Supervised Pre-Training (Electricity) ===")
seq_len, batch_size = 512, 32
train_dataset_elec_ssl = WeatherDataset(train_df_elec[elec_features], seq_len, pred_len=96)
val_dataset_elec_ssl = WeatherDataset(val_df_elec[elec_features], seq_len, pred_len=96)
train_loader_elec_ssl = DataLoader(train_dataset_elec_ssl, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader_elec_ssl = DataLoader(val_dataset_elec_ssl, batch_size=batch_size, shuffle=False)

model_ssl_elec = PatchTST_SelfSupervised(num_features=len(elec_features), seq_len=seq_len, patch_len=12, stride=12)
trained_ssl_model_elec = pretrain_model_with_es(model_ssl_elec, train_loader_elec_ssl, val_loader_elec_ssl, epochs=100, lr=1e-4, patience=10)

print("\n=== PHASE 2: Linear Probing (Electricity) ===")
results_summary_elec = {"Horizon": [], "Val MSE": [], "Val MAE": []}
for pred_len in [96, 192, 336, 720]:
    print(f"\n--- Horizon: T={pred_len} ---")
    train_loader = DataLoader(WeatherDataset(train_df_elec[elec_features], seq_len, pred_len), batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(WeatherDataset(val_df_elec[elec_features], seq_len, pred_len), batch_size=batch_size, shuffle=False)

    model_sup_elec = PatchTST_Supervised(num_features=len(elec_features), seq_len=seq_len, pred_len=pred_len, patch_len=12, stride=12)
    trained_sup_model_elec = linear_probing_with_es(model_sup_elec, trained_ssl_model_elec, train_loader, val_loader, epochs=20, lr=1e-4, patience=5)

    trained_sup_model_elec.eval()
    v_mse, v_mae = 0.0, 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            preds = trained_sup_model_elec(bx)
            v_mse += nn.MSELoss()(preds, by).item()
            v_mae += nn.L1Loss()(preds, by).item()

    results_summary_elec["Horizon"].append(pred_len)
    results_summary_elec["Val MSE"].append(v_mse / len(val_loader))
    results_summary_elec["Val MAE"].append(v_mae / len(val_loader))

print("\nFINAL RESULTS (ELECTRICITY)")
print(pd.DataFrame(results_summary_elec).set_index("Horizon").round(3))

=== PREPROCESSING ELECTRICITY DATA ===

=== PHASE 1: Self-Supervised Pre-Training (Electricity) ===


Epoch 1/100 [Pre-train]: 100%|██████████| 3049/3049 [46:10<00:00,  1.10it/s]


Epoch 1 | Train Loss: 0.1852 | Val Loss: 0.1169


Epoch 2/100 [Pre-train]: 100%|██████████| 3049/3049 [46:44<00:00,  1.09it/s]


Epoch 2 | Train Loss: 0.0615 | Val Loss: 0.0553


...

Epoch 13/100 [Pre-train]: 100%|██████████| 3049/3049 [46:44<00:00,  1.09it/s]


Epoch 13 | Train Loss: 0.0000 | Val Loss: 0.0217
EarlyStopping counter: 7 out of 10


Epoch 14/100 [Pre-train]: 100%|██████████| 3049/3049 [46:44<00:00,  1.09it/s]


Epoch 14 | Train Loss: 0.0000 | Val Loss: 0.0217
EarlyStopping counter: 8 out of 10


Epoch 15/100 [Pre-train]:  37%|███▋      | 1123/3049 [17:12<29:32,  1.09it/s]